# CivilizationOS — Council Voice LoRA Fine-Tune

Fine-tunes **Qwen2.5 3B** with Unsloth LoRA on the synthetic council-voice dataset.
Runs on **Google Colab free T4 GPU** (15 GB VRAM). Exports to GGUF and can be loaded
into Ollama as a custom model, cutting Tier-2 (Claude) spend to near zero.

**Steps**
1. Install deps
2. Load model with Unsloth 4-bit quant
3. Load + format dataset
4. Train with SFTTrainer
5. Eval persona-consistency gate
6. Export GGUF
7. Upload to HuggingFace Hub (optional)

**MLflow** experiment tracking writes to `mlruns/` (locally) or a remote URI.

In [ ]:
# 1. Install — takes ~3 min on Colab
!pip install -q unsloth mlflow datasets trl
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
import json, os, mlflow
from pathlib import Path
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME   = 'Qwen/Qwen2.5-3B-Instruct'
OUTPUT_DIR   = '/content/council_lora'
GGUF_DIR     = '/content/council_gguf'
DATASET_PATH = 'ml/dataset/council_voices.jsonl'   # upload to Colab or mount Drive
MAX_SEQ_LEN  = 1024
LORA_RANK    = 16

# MLflow — tracks this run locally; copy mlruns/ back to project after training
mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('council_lora')
print('Setup complete')

In [ ]:
# 2. Load model + tokenizer (4-bit quant via Unsloth — fits in 15 GB T4)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # auto-detect
    load_in_4bit=True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# 3. Load + format dataset
raw = [json.loads(l) for l in open(DATASET_PATH)]

def format_sample(s):
    # ChatML format Qwen2.5 expects
    return {
        'text': (
            '<|im_start|>system\n' + s['system'] + '<|im_end|>\n'
            '<|im_start|>user\n'   + s['user']   + '<|im_end|>\n'
            '<|im_start|>assistant\n' + s['assistant'] + '<|im_end|>'
        )
    }

ds = Dataset.from_list([format_sample(s) for s in raw])
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

In [ ]:
# 4. Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',   # we log to MLflow manually below
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

with mlflow.start_run(run_name='qwen2.5-3b-council-lora') as run:
    mlflow.log_params({
        'base_model': MODEL_NAME,
        'lora_rank': LORA_RANK,
        'epochs': training_args.num_train_epochs,
        'lr': training_args.learning_rate,
        'train_samples': len(train_ds),
    })

    train_result = trainer.train()

    for k, v in train_result.metrics.items():
        mlflow.log_metric(k, v)

    run_id = run.info.run_id
    print(f'Training complete. MLflow run: {run_id}')

In [ ]:
# 5. Persona-consistency eval gate (inline, no Ollama needed here)
import re

ROLE_PATTERNS = {
    'Historian':   [r'\bprecedent\b', r'\bhistoric\b', r'\brecord\b', r'\blast (time|year)\b'],
    'Strategist':  [r'\bintervention\b', r'\bdeploy\b', r'\bimmediately\b', r'\bactionable\b'],
    'Skeptic':     [r'\brisk\b', r'\bassumption\b', r'\bunintended\b', r'\bsafeguard\b'],
    'Predictor':   [r'\b\d+%\b', r'\bforecast\b', r'\bmost likely\b', r'\bworst.case\b'],
    'Synthesizer': [r'\bVERDICT:\b', r'\bsuccess metric\b', r'\bexecutes?\b'],
}

FastLanguageModel.for_inference(model)
results = {}
for sample in eval_ds.select(range(min(30, len(eval_ds)))):
    text = sample['text']
    # extract role from system prompt
    role_match = re.search(r'council (\w+)\.', text)
    if not role_match:
        continue
    role = role_match.group(1)
    # generate
    inputs = tokenizer(text.split('<|im_start|>assistant')[0] + '<|im_start|>assistant\n',
                       return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.01)
    response = tokenizer.decode(out[0], skip_special_tokens=True).split('assistant\n')[-1]
    patterns = ROLE_PATTERNS.get(role, [])
    score = sum(1 for p in patterns if re.search(p, response, re.I)) / max(len(patterns), 1)
    results.setdefault(role, []).append(score)

print('\n── Persona eval ──')
all_pass = True
with mlflow.start_run(run_id=run_id):
    for role, scores in results.items():
        avg = sum(scores) / len(scores)
        passed = avg >= 0.4
        all_pass = all_pass and passed
        mlflow.log_metric(f'eval_{role}_persona', avg)
        print(f'  {role:<12} {avg:.2f}  {"PASS" if passed else "FAIL"}')
    mlflow.log_metric('promoted', int(all_pass))
print(f'\nModel promoted: {all_pass}')

In [ ]:
# 6. Export to GGUF (Q4_K_M quantization — good quality/size tradeoff for Ollama)
if all_pass:
    model.save_pretrained_gguf(
        GGUF_DIR,
        tokenizer,
        quantization_method='q4_k_m',
    )
    print(f'GGUF saved to {GGUF_DIR}')
    print('\nTo load in Ollama:')
    print('  1. Copy the .gguf file to your laptop')
    print('  2. Create a Modelfile:')
    print('     FROM ./council_voice_q4.gguf')
    print('     PARAMETER temperature 0.7')
    print('  3. ollama create council-voice -f Modelfile')
    print('  4. Set OLLAMA_COUNCIL_MODEL=council-voice in .env')
else:
    print('Model failed persona gate — NOT exported. Re-train with more data or epochs.')

In [ ]:
# 7. (Optional) Push adapter to HuggingFace Hub
# HF_TOKEN = 'hf_...'  # set your token
# model.push_to_hub('your-hf-username/civos-council-lora', token=HF_TOKEN)
# tokenizer.push_to_hub('your-hf-username/civos-council-lora', token=HF_TOKEN)
print('Done! Download mlruns/ and the .gguf from Colab before the session expires.')